# Knowledge Time in SkyPortal

A historical state is not defined only by what physically happened. It is
defined by what had entered the record by a chosen cutoff. This notebook tests
whether the raw SkyPortal capture supplies that second timestamp for each fact
type used by the project.

The analysis reads raw captures only and follows **question, measurement,
decision**.


In [1]:
from pathlib import Path
from datetime import datetime
import json

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DETAIL_ROOT = ROOT / "data" / "raw" / "skyportal" / "source_detail_20260724"
INVENTORY_ROOT = ROOT / "data" / "raw" / "skyportal" / "inventory"
CAPTURE_STAMP = "20260720"
PROFILES = ["grandma_base", "gcn", "ep", "grb"]


def load_detail_records(collection):
    records = []
    files = []
    for source_directory in sorted(path for path in DETAIL_ROOT.iterdir() if path.is_dir()):
        path = source_directory / f"{collection}.json"
        if not path.exists():
            continue
        wrapper = json.loads(path.read_text(encoding="utf-8"))
        payload_data = wrapper.get("payload", {}).get("data")
        if collection == "followup_requests":
            rows = payload_data.get("followup_requests", []) if isinstance(payload_data, dict) else []
        elif collection == "spectra":
            rows = payload_data.get("spectra", []) if isinstance(payload_data, dict) else []
        else:
            rows = payload_data if isinstance(payload_data, list) else []
        records.extend(
            {"source_id": source_directory.name, "record": row}
            for row in rows if isinstance(row, dict)
        )
        files.append(path)
    return records, files


detail_records = {}
detail_files = {}
for collection in ["comments", "photometry", "followup_requests"]:
    detail_records[collection], detail_files[collection] = load_detail_records(collection)

inventory_directories = {}
listing_rows = []
records_by_source = {}
for profile in PROFILES:
    matches = sorted(INVENTORY_ROOT.glob(f"source_inventory_{profile}_{CAPTURE_STAMP}_*"))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one frozen {profile} inventory, found {len(matches)}")
    inventory_directories[profile] = matches[0]
    for page_path in sorted(matches[0].glob("sources_page_*.json")):
        payload = json.loads(page_path.read_text(encoding="utf-8"))
        rows = [
            row for row in payload.get("data", {}).get("sources", [])
            if isinstance(row, dict)
        ]
        listing_rows.extend(rows)
        for row in rows:
            source_id = row.get("id")
            if isinstance(source_id, str):
                records_by_source.setdefault(source_id, {})[profile] = row

def is_non_empty(value):
    return value is not None and value != "" and value != [] and value != {}

selected_listing = {
    source_id: max(
        profile_rows.items(),
        key=lambda item: (
            sum(is_non_empty(value) for value in item[1].values()),
            item[0],
        ),
    )[1]
    for source_id, profile_rows in records_by_source.items()
}

input_summary = pd.DataFrame(
    [
        ("comments", len(detail_files["comments"]), len(detail_records["comments"])),
        ("photometry", len(detail_files["photometry"]), len(detail_records["photometry"])),
        ("followup_requests", len(detail_files["followup_requests"]), len(detail_records["followup_requests"])),
        ("listing records", sum(len(list(path.glob("sources_page_*.json"))) for path in inventory_directories.values()), len(listing_rows)),
        ("deduplicated listing sources", 4, len(selected_listing)),
    ],
    columns=["raw_input", "files", "records"],
)
print(input_summary.to_string(index=False))


                   raw_input  files  records
                    comments    799     2950
                  photometry    799     7968
           followup_requests    800     2359
             listing records     13      982
deduplicated listing sources      4      800


## 1. The question

**QUESTION.** What timestamp is required to reconstruct “what was known at
time $T$”?


In [2]:
reconstruction_requirements = pd.DataFrame(
    [
        ("occurrence time", "When the event or measurement physically happened"),
        ("knowledge time", "When the fact entered the available record"),
        ("cutoff rule", "Keep a fact only when knowledge_time <= T"),
    ],
    columns=["requirement", "meaning"],
)
print(reconstruction_requirements.to_string(index=False))


    requirement                                           meaning
occurrence time When the event or measurement physically happened
 knowledge time        When the fact entered the available record
    cutoff rule         Keep a fact only when knowledge_time <= T


**DECISION.** State reconstruction requires a knowledge-time for every fact.
Observation time alone is insufficient because a measurement may be added to
SkyPortal long after it occurred.


## 2. Date fields by fact type

**QUESTION.** Which actual fields carry knowledge, occurrence, planning, or
modification time in one populated example of each required fact type?


In [3]:
def flatten_leaves(value, path=""):
    if isinstance(value, dict):
        for key, child in value.items():
            child_path = f"{path}.{key}" if path else key
            yield from flatten_leaves(child, child_path)
    elif isinstance(value, list):
        for index, child in enumerate(value):
            yield from flatten_leaves(child, f"{path}[{index}]")
    else:
        yield path, value


def parses_as_iso_date(value):
    if not isinstance(value, str):
        return False
    try:
        datetime.fromisoformat(value.replace("Z", "+00:00"))
    except ValueError:
        return False
    return True


def detail_example(collection, source_id):
    return next(
        item["record"] for item in detail_records[collection]
        if item["source_id"] == source_id
    )


grb_listing = selected_listing["GRB241030"]
grb_gcn_photometry = [
    item["record"] for item in detail_records["photometry"]
    if item["source_id"] == "GRB241030"
    and item["record"].get("origin") == "GCN"
]
mjd_epoch = pd.Timestamp("1858-11-17", tz="UTC")
for row in grb_gcn_photometry:
    observation_time = mjd_epoch + pd.to_timedelta(float(row["mjd"]), unit="D")
    row["_knowledge_lag_hours"] = (
        pd.to_datetime(row["created_at"], utc=True) - observation_time
    ).total_seconds() / 3600.0
photometry_example = max(
    grb_gcn_photometry, key=lambda row: row["_knowledge_lag_hours"]
)

examples = {
    "comment": ("2026owq", detail_example("comments", "2026owq")),
    "classification": ("GRB241030", grb_listing["classifications"][0]),
    "photometry": ("GRB241030", photometry_example),
    "redshift_version": ("GRB241030", grb_listing["redshift_history"][0]),
    "summary_version": ("GRB241030", grb_listing["summary_history"][0]),
    "followup_request": ("2026owq", detail_example("followup_requests", "2026owq")),
}

# Every leaf is enumerated first. For follow-up requests, the fact-local view is
# the request root plus payload; expanded obj/allocation/user objects are linked
# context and are counted separately rather than attributed to the request.
linked_followup_roots = {"obj", "allocation", "target_groups", "requester", "watchers"}
local_leaves = {}
inventory_rows = []
for fact_type, (source_id, example) in examples.items():
    all_leaves = list(flatten_leaves(example))
    if fact_type == "followup_request":
        fact_leaves = [
            (path, value) for path, value in all_leaves
            if path.split(".", 1)[0].split("[", 1)[0] not in linked_followup_roots
        ]
    else:
        fact_leaves = all_leaves
    local_leaves[fact_type] = fact_leaves
    inventory_rows.append(
        {
            "fact_type": fact_type,
            "source_id": source_id,
            "all_leaf_fields": len(all_leaves),
            "fact_local_leaf_fields": len(fact_leaves),
        }
    )

time_roles = {
    ("comment", "created_at"): "knowledge_time",
    ("comment", "modified"): "modification_time",
    ("classification", "created_at"): "knowledge_time",
    ("classification", "modified"): "modification_time",
    ("photometry", "mjd"): "occurrence_time",
    ("photometry", "created_at"): "knowledge_time",
    ("redshift_version", "set_at_utc"): "knowledge_time",
    ("summary_version", "set_at_utc"): "knowledge_time",
    ("followup_request", "created_at"): "knowledge_time",
    ("followup_request", "modified"): "modification_time",
    ("followup_request", "payload.start_date"): "planned_window_start",
    ("followup_request", "payload.end_date"): "planned_window_end",
}

date_rows = []
for fact_type, (source_id, _) in examples.items():
    for field_path, value in local_leaves[fact_type]:
        is_mjd = fact_type == "photometry" and field_path == "mjd"
        if parses_as_iso_date(value) or is_mjd:
            date_rows.append(
                {
                    "fact_type": fact_type,
                    "source_id": source_id,
                    "field_path": field_path,
                    "value": value,
                    "time_role": time_roles.get((fact_type, field_path), "unclassified"),
                }
            )

date_examples = pd.DataFrame(date_rows)
if (date_examples["time_role"] == "unclassified").any():
    raise AssertionError("An exhaustively discovered date field lacks a time role")

print("Exhaustive leaf enumeration:")
print(pd.DataFrame(inventory_rows).to_string(index=False))
print("Date-bearing fields discovered from those leaves:")
print(date_examples.to_string(index=False))


Exhaustive leaf enumeration:
       fact_type source_id  all_leaf_fields  fact_local_leaf_fields
         comment   2026owq               10                      10
  classification GRB241030               11                      11
      photometry GRB241030               28                      28
redshift_version GRB241030                5                       5
 summary_version GRB241030                5                       5
followup_request   2026owq              408                      19
Date-bearing fields discovered from those leaves:
       fact_type source_id         field_path                      value            time_role
         comment   2026owq         created_at 2026-06-11T08:08:33.407198       knowledge_time
         comment   2026owq           modified 2026-06-11T08:08:33.407198    modification_time
  classification GRB241030         created_at 2024-10-30T05:52:42.096180       knowledge_time
  classification GRB241030           modified 2024-10-30T05:52:42.096

**DECISION.** Comments, classifications, photometry, and follow-up requests use
`created_at` as knowledge-time. Redshift and summary versions use
`set_at_utc`. Photometry separately carries occurrence time in `mjd`;
follow-up requests also carry a planned date window; and `modified` records
later object changes rather than first knowledge.


## 3. Knowledge-time coverage

**QUESTION.** Among the facts present in the raw capture, how often is the
required knowledge-time populated?


In [ ]:
facts = {
    "comment": [item["record"] for item in detail_records["comments"]],
    "classification": [
        fact for source in selected_listing.values()
        for fact in (source.get("classifications") or [])
    ],
    "photometry": [item["record"] for item in detail_records["photometry"]],
    "redshift_version": [
        fact for source in selected_listing.values()
        for fact in (source.get("redshift_history") or [])
    ],
    "summary_version": [
        fact for source in selected_listing.values()
        for fact in (source.get("summary_history") or [])
    ],
    "followup_request": [item["record"] for item in detail_records["followup_requests"]],
}
knowledge_fields = {
    "comment": "created_at",
    "classification": "created_at",
    "photometry": "created_at",
    "redshift_version": "set_at_utc",
    "summary_version": "set_at_utc",
    "followup_request": "created_at",
}

coverage_rows = []
for fact_type, records in facts.items():
    knowledge_field = knowledge_fields[fact_type]
    with_time = sum(
        record.get(knowledge_field) not in (None, "") for record in records
    )
    coverage_rows.append(
        {
            "fact_type": fact_type,
            "knowledge_field": knowledge_field,
            "records": len(records),
            "with_knowledge_time": with_time,
            "coverage_pct": 100.0 * with_time / len(records) if records else 0.0,
        }
    )
knowledge_coverage = pd.DataFrame(coverage_rows)

control_rows = []
for source_id, expected_comments, expected_followups in [
    ("2026owq", 92, 9),
    ("GRB241030", 57, 8),
]:
    observed_comments = sum(
        item["source_id"] == source_id for item in detail_records["comments"]
    )
    observed_followups = sum(
        item["source_id"] == source_id for item in detail_records["followup_requests"]
    )
    control_rows.append(
        {
            "source_id": source_id,
            "comments": observed_comments,
            "expected_comments": expected_comments,
            "followup_requests": observed_followups,
            "expected_followups": expected_followups,
            "matches": (
                observed_comments == expected_comments
                and observed_followups == expected_followups
            ),
        }
    )
control_counts = pd.DataFrame(control_rows)

print(knowledge_coverage.to_string(
    index=False, formatters={"coverage_pct": lambda value: f"{value:.2f}%"}
))
print("Control-source counts:")
print(control_counts.to_string(index=False))


       fact_type knowledge_field  records  with_knowledge_time coverage_pct
         comment      created_at     2950                 2950      100.00%
  classification      created_at      416                  416      100.00%
      photometry      created_at     7968                 7968      100.00%
redshift_version      set_at_utc       83                   83      100.00%
 summary_version      set_at_utc     1274                 1274      100.00%
followup_request      created_at     2359                 2359      100.00%
Control-source counts:
source_id  comments  expected_comments  followup_requests  expected_followups  matches
  2026owq        92                 92                  9                   9     True
GRB241030        57                 57                  8                   8     True


**FINDING.** Knowledge-time coverage is **100% for all six observed fact
types**: 2,950 comments, 416 classifications, 7,968 photometry rows, 83
redshift versions, 1,274 summary versions, and 2,359 follow-up requests. The
2026owq and GRB241030 comment/follow-up controls match exactly.


## 4. Observation time is not knowledge time

**QUESTION.** Can one fact carry both a physical observation timestamp and a
later database-entry timestamp?


In [5]:
observation_time = mjd_epoch + pd.to_timedelta(
    float(photometry_example["mjd"]), unit="D"
)
knowledge_time = pd.to_datetime(photometry_example["created_at"], utc=True)
knowledge_lag_hours = (knowledge_time - observation_time).total_seconds() / 3600.0
lag_example = pd.DataFrame(
    [{
        "source_id": "GRB241030",
        "photometry_id": photometry_example["id"],
        "origin": photometry_example["origin"],
        "filter": photometry_example["filter"],
        "mjd": photometry_example["mjd"],
        "observation_time_utc": observation_time.isoformat(),
        "created_at": photometry_example["created_at"],
        "knowledge_lag_hours": knowledge_lag_hours,
    }]
)
print(lag_example.to_string(
    index=False, formatters={"knowledge_lag_hours": lambda value: f"{value:.3f}"}
))


source_id  photometry_id origin      filter        mjd                observation_time_utc                 created_at knowledge_lag_hours
GRB241030          37670    GCN uvot::white 60613.2435 2024-10-30T05:50:38.399999732+00:00 2024-10-30T16:56:27.973714              11.097


**FINDING.** GRB241030 photometry row 37670 was observed at
**2024-10-30 05:50:38 UTC** (`mjd=60613.2435`) and entered SkyPortal at
**16:56:27 UTC**, **11.097 h later**. `mjd` answers when the measurement
occurred; `created_at` answers when it became available in SkyPortal.


## 5. Decision

**QUESTION.** Does the raw SkyPortal capture support causal truncation for
historical state reconstruction?


In [6]:
all_fact_types_have_knowledge_time = bool(
    (knowledge_coverage["coverage_pct"] == 100.0).all()
)
truncation_rule = pd.DataFrame(
    [
        ("SkyPortal-native fact", "created_at or set_at_utc <= cutoff T"),
        ("Physical ordering", "occurrence time such as mjd"),
        ("Circular-origin fact", "publication time deferred to NB04/NB05"),
    ],
    columns=["case", "temporal rule"],
)
print(f"All observed fact types have knowledge-time: {all_fact_types_have_knowledge_time}")
print(truncation_rule.to_string(index=False))


All observed fact types have knowledge-time: True
                 case                          temporal rule
SkyPortal-native fact   created_at or set_at_utc <= cutoff T
    Physical ordering            occurrence time such as mjd
 Circular-origin fact publication time deferred to NB04/NB05


**DECISION.** Causal truncation is possible for the six observed SkyPortal fact
types because every record carries `created_at` or `set_at_utc`. Historical
state construction will key on that knowledge-time, not on occurrence time.

One provenance nuance remains: when a fact was transcribed from a GCN circular,
the honest knowledge-time is the circular's publication time rather than the
later SkyPortal `created_at`. That correction is deferred to NB04/NB05.


## Decisions summary

The table records the evidence-backed temporal decisions established here.


In [7]:
decisions = pd.DataFrame(
    [
        ("Require knowledge-time for state reconstruction", "Occurrence alone cannot define what was known", 1),
        ("Use created_at for row facts", "Comments, classifications, photometry, follow-ups", 2),
        ("Use set_at_utc for version facts", "Redshift and summary histories", 2),
        ("Treat occurrence and knowledge as separate", "GRB241030 example differs by 11.097 h", 4),
        ("Truncate on knowledge-time", "100% coverage across six observed fact types", 5),
        ("Defer circular provenance correction", "Publication predates SkyPortal transcription", 5),
    ],
    columns=["decision", "evidence", "section"],
)
print(decisions.to_string(index=False))


                                       decision                                          evidence  section
Require knowledge-time for state reconstruction     Occurrence alone cannot define what was known        1
                   Use created_at for row facts Comments, classifications, photometry, follow-ups        2
               Use set_at_utc for version facts                    Redshift and summary histories        2
     Treat occurrence and knowledge as separate             GRB241030 example differs by 11.097 h        4
                     Truncate on knowledge-time      100% coverage across six observed fact types        5
           Defer circular provenance correction      Publication predates SkyPortal transcription        5


**DECISION.** The raw capture contains the timestamps required to construct
causal past states. The next notebooks can measure lag and correct provenance;
they do not need to invent a knowledge-time field.
